# MIC Analysis — TEM-1 Validation Experiments

**13 TEM-1 variants** (8 designed + 4 colony isolates + DD negative control) tested against **AZT** and **AMP** via plate-reader growth curves.

Pipeline: raw XLSX → background subtraction → AUC integration → normalize by no-drug control → interpolate MIC at 5% threshold → cross-reference with epistasis fitness landscape.

In [ ]:
import sys
from pathlib import Path
import json

import polars as pl
import numpy as np
from scipy import stats
import altair as alt

# Paths
ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
FIGURES_DIR = ROOT.parent / "sadik" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
EPISTASIS_PARQUET = ROOT.parent / "Gaszek_Yildiz_Meng_2025" / "data" / "processed" / "Epistasis_Combined.parquet"

# Import Deniz's utils
sys.path.insert(0, str(ROOT / "plate_reader"))
import utils as utl

alt.data_transformers.disable_max_rows()
print(f"Data dir: {DATA_DIR}")
print(f"Epistasis parquet: {EPISTASIS_PARQUET.exists()}")

In [ ]:
# ── Load metadata ──
with open(ROOT / "variants.json") as f:
    variants_meta = json.load(f)["variants"]

with open(ROOT / "experiments.json") as f:
    experiments_raw = json.load(f)["experiments"]

VARIANT_NAMES = list(variants_meta.keys())
print(f"{len(VARIANT_NAMES)} variants: {VARIANT_NAMES}")

# ── Adapt experiments.json → Deniz's utils dict format ──
BG_EXCLUDE = {"AZT": ["A16"], "AMP": ["A8"]}

def adapt_experiment(exp_raw: dict) -> dict:
    """Convert experiments.json entry to the dict format Deniz's utils expects."""
    drug_abbrev = exp_raw["drug_abbrev"]
    return {
        "file_name": str(DATA_DIR / exp_raw["file_name"]),
        "drug": exp_raw["drug"],
        "largest_conc": exp_raw["largest_conc_ugml"],
        "dilution": exp_raw["dilution_fold"],
        "plates": [[p["variant_1"], p["variant_2"]] for p in exp_raw["plates"]],
        "variants": [
            exp_raw["plate_layout"]["variant_1_rows"],
            exp_raw["plate_layout"]["variant_2_rows"],
        ],
        "background": [
            exp_raw["plate_layout"]["background_rows"],
            list(range(2, 13)),
            BG_EXCLUDE[drug_abbrev],
        ],
    }

experiments = {e["drug_abbrev"]: adapt_experiment(e) for e in experiments_raw}
for drug, exp in experiments.items():
    print(f"{drug}: {len(exp['plates'])} plates, largest_conc={exp['largest_conc']} µg/mL")

In [ ]:
# ── Load epistasis fitness data ──
# Build genotype lookup: variant_name → genotype_13
GENO_LOOKUP = {name: v["genotype_13"] for name, v in variants_meta.items() if v["genotype_13"] is not None}
NAME_LOOKUP = {v: k for k, v in GENO_LOOKUP.items()}
print(f"{len(GENO_LOOKUP)} variants with genotype_13 (DD excluded)")

WT_GENOTYPE = variants_meta["WT"]["genotype_13"]  # "LQMERMAGERTRN"


def variant_label(name: str) -> str:
    """Dot-notation label: matching positions → '.', mismatches keep letter, + (short_name)."""
    if name == "DD":
        return "DD"
    geno = GENO_LOOKUP.get(name)
    if geno is None:
        return name
    if name == "WT":
        return f"{geno} ({name})"
    dots = "".join("." if g == w else g for g, w in zip(geno, WT_GENOTYPE))
    return f"{dots} ({name})"


# Raw name order — LABEL_ORDER computed in cell 7 after VARIANT_COLORS defined
_NAME_ORDER = [
    "WT", "v1.1", "v1.2", "v1.3", "v2.1", "v2.2", "v2.3", "v3.1",
    "Col1", "Col2", "Col3", "Col4", "DD",
]

# Load only the rows we need
target_genotypes = list(GENO_LOOKUP.values())
epi_df = (
    pl.scan_parquet(EPISTASIS_PARQUET)
    .filter(pl.col("Genotype").is_in(target_genotypes))
    .select("Genotype", "Epistatic Order", "Drug", "Concentration", "Fitness")
    .collect()
)
print(f"Epistasis rows loaded: {len(epi_df)}")
print(f"Unique genotypes: {epi_df['Genotype'].n_unique()}")

# Concentration ranges in epistasis dataset
for drug in ["AZT", "AMP"]:
    concs = epi_df.filter(pl.col("Drug") == drug)["Concentration"].unique().sort().to_list()
    print(f"  {drug} concs: {concs}")

# Preview labels
print("\nVariant labels:")
for name in _NAME_ORDER:
    print(f"  {name:6s} → {variant_label(name)}")

In [ ]:
# ── Process all plates → long-form DataFrame ──
# Columns 1-11 are drug dilution series (col 11 = 0 µg/mL = no-drug control)
# Column 12 is empty/unused — excluded entirely

rows = []

for drug_abbrev, experiment in experiments.items():
    concs, _, _ = utl.form_concentrations(experiment)  # 11 values, last = 0.0
    n_plates = len(experiment["plates"])

    for plate_idx in range(n_plates):
        plate_variants = experiment["plates"][plate_idx]  # [v1_name, v2_name]
        variant_rows_list = experiment["variants"]  # [[B,C,D], [E,F,G]]

        # Read and process plate
        df_raw = utl.read_plate(experiment, plate_idx)
        df_proc = utl.subtract_bg_and_integrate(experiment, df_raw, drop_first=3)

        # For final OD: mean of last 5 bg-subtracted timepoints
        n_rows_data = len(df_proc)
        last5_slice = slice(max(0, n_rows_data - 5), n_rows_data)

        for slot_idx, variant_name in enumerate(plate_variants):
            letters = variant_rows_list[slot_idx]  # e.g. ['B','C','D']

            for col_idx in range(11):  # columns 1-11 only
                col_num = col_idx + 1
                concentration = concs[col_idx]
                is_control = concentration == 0.0  # column 11

                for rep_idx, letter in enumerate(letters):
                    well = f"{letter}{col_num}"
                    auc_col = f"{well}_auc"
                    bgsub_col = f"{well}_bgsub"

                    if auc_col not in df_proc.columns:
                        continue

                    # Final AUC = last value of cumulative integral
                    final_auc = df_proc[auc_col][-1]

                    # Final OD = mean of last 5 bg-subtracted timepoints
                    final_od = df_proc[bgsub_col][last5_slice].mean()

                    rows.append({
                        "drug": drug_abbrev,
                        "plate": plate_idx + 1,
                        "variant": variant_name,
                        "concentration": concentration,
                        "is_control": is_control,
                        "replicate": rep_idx,
                        "well": well,
                        "final_auc": float(final_auc),
                        "final_od": float(final_od),
                    })

long_df = pl.DataFrame(rows)
print(f"Long DataFrame: {long_df.shape}")
print(f"Variants: {long_df['variant'].unique().sort().to_list()}")
print(f"Drugs: {long_df['drug'].unique().to_list()}")

# Sanity: control wells should have positive AUC (cells grow without drug)
ctrl = long_df.filter(pl.col("is_control"))
print(f"\nControl (no-drug) AUC stats:")
print(f"  min={ctrl['final_auc'].min():.2f}, median={ctrl['final_auc'].median():.2f}, max={ctrl['final_auc'].max():.2f}")

# Plate 7 has v1.1 in both slots → 6 replicates
v11_counts = long_df.filter((pl.col('variant') == 'v1.1') & ~pl.col('is_control')).group_by('drug', 'concentration').len()
print(f"v1.1 replicates per drug×conc: {v11_counts['len'].unique().to_list()}")

In [ ]:
# ── Compute MIC ──
# Method (Sadik): normalize AUC by no-drug control median → interpolate conc at 5% threshold
# Interpolation: reverse dose-response arrays so norm_auc goes ascending (high conc → low conc),
# then np.interp finds where it crosses the threshold.

MIC_THRESHOLD = 0.05

mic_records = []
dr_records = []  # dose-response for plotting

for (drug, variant), grp in long_df.group_by(["drug", "variant"]):
    # Control median AUC (column 11, concentration = 0)
    ctrl = grp.filter(pl.col("is_control"))
    control_median = ctrl["final_auc"].median()

    if control_median is None or control_median <= 0:
        print(f"  WARN: {drug} {variant} control_median={control_median:.4f}, skipping")
        continue

    # Drug wells only (not control)
    drug_wells = grp.filter(~pl.col("is_control"))

    # Normalize AUC
    drug_wells = drug_wells.with_columns(
        (pl.col("final_auc") / control_median).alias("norm_auc")
    )

    # Per-concentration stats
    conc_stats = (
        drug_wells
        .group_by("concentration")
        .agg(
            pl.col("norm_auc").mean().alias("norm_auc_mean"),
            pl.col("norm_auc").std().alias("norm_auc_std"),
            pl.col("norm_auc").count().alias("n_reps"),
        )
        .sort("concentration")
    )

    concs_arr = conc_stats["concentration"].to_numpy()
    means_arr = conc_stats["norm_auc_mean"].to_numpy()
    stds_arr = conc_stats["norm_auc_std"].to_numpy()

    # Dose-response records for plotting (non-zero concs only)
    for i in range(len(concs_arr)):
        if concs_arr[i] == 0.0:
            continue
        std_val = stds_arr[i] if stds_arr[i] is not None and not np.isnan(stds_arr[i]) else 0.0
        dr_records.append({
            "drug": drug,
            "variant": variant,
            "concentration": concs_arr[i],
            "norm_auc_mean": means_arr[i],
            "norm_auc_std": std_val,
            "norm_auc_lo": means_arr[i] - std_val,
            "norm_auc_hi": means_arr[i] + std_val,
        })

    # MIC interpolation (Sadik's method)
    # Filter to non-zero concs, sorted ascending by concentration
    mask = concs_arr > 0
    concs_drug = concs_arr[mask]
    means_drug = means_arr[mask]

    if len(concs_drug) > 1:
        # Reverse: means go from high-conc (low growth) to low-conc (high growth) = ascending
        # np.interp finds where reversed means cross threshold
        mic_val = np.interp(
            MIC_THRESHOLD,
            means_drug[::-1],   # xp: ascending (low→high norm_auc)
            concs_drug[::-1],   # fp: descending (high→low concentration)
        )
    else:
        mic_val = np.nan

    mic_records.append({
        "drug": drug,
        "variant": variant,
        "mic_ugml": float(mic_val),
        "control_median_auc": float(control_median),
    })

mic_df = pl.DataFrame(mic_records)
dr_df = pl.DataFrame(dr_records)

# Add genotype_13 label column
dr_df = dr_df.with_columns(
    pl.col("variant").map_elements(variant_label, return_dtype=pl.Utf8).alias("label")
)

print(f"MIC table: {mic_df.shape}")
print(mic_df.sort("drug", "mic_ugml", descending=[False, True]))

# ── Per-replicate MIC (for dot plot error bars) ──
mic_rep_records = []
for (drug, variant), grp in long_df.group_by(["drug", "variant"]):
    ctrl = grp.filter(pl.col("is_control"))
    control_median = ctrl["final_auc"].median()
    if control_median is None or control_median <= 0:
        continue

    drug_wells = grp.filter(~pl.col("is_control")).with_columns(
        pl.col("well").str.slice(0, 1).alias("rep_letter"),
        (pl.col("final_auc") / control_median).alias("norm_auc"),
    )

    for (rep_letter,), rep_grp in drug_wells.group_by("rep_letter"):
        rep_conc = rep_grp.sort("concentration")
        concs_arr = rep_conc["concentration"].to_numpy()
        vals_arr = rep_conc["norm_auc"].to_numpy()
        mask = concs_arr > 0
        if mask.sum() > 1:
            mic_val = np.interp(MIC_THRESHOLD, vals_arr[mask][::-1], concs_arr[mask][::-1])
        else:
            mic_val = np.nan
        mic_rep_records.append({
            "drug": drug, "variant": variant,
            "rep_letter": rep_letter,
            "mic_ugml": float(mic_val),
        })

mic_rep_df = pl.DataFrame(mic_rep_records).with_columns(
    pl.col("mic_ugml").clip(lower_bound=1e-3).log(base=10).alias("log10_mic"),
    pl.col("variant").map_elements(variant_label, return_dtype=pl.Utf8).alias("label"),
)
print(f"\nPer-replicate MIC table: {mic_rep_df.shape}")
print(f"Replicates per variant×drug:")
print(mic_rep_df.group_by("variant", "drug").len().sort("variant", "drug"))

In [ ]:
# ── Cell A: Time-to-Threshold (TTT) ──
# Metric: time for bg-subtracted OD to first reach 0.2 per well.
# Censored wells (OD never ≥ 0.2) get ttt = t_max, flagged is_censored=True.

TTT_THRESHOLD = 0.2

ttt_rows = []

for drug_abbrev, experiment in experiments.items():
    concs, _, _ = utl.form_concentrations(experiment)
    n_plates = len(experiment["plates"])

    for plate_idx in range(n_plates):
        plate_variants = experiment["plates"][plate_idx]
        variant_rows_list = experiment["variants"]

        df_raw = utl.read_plate(experiment, plate_idx)
        df_proc = utl.subtract_bg_and_integrate(experiment, df_raw, drop_first=3)
        t_hours = df_proc["t_hours"].to_numpy().astype(float)
        t_max = float(t_hours[-1])

        for slot_idx, variant_name in enumerate(plate_variants):
            letters = variant_rows_list[slot_idx]

            for col_idx in range(11):
                col_num = col_idx + 1
                concentration = concs[col_idx]
                if concentration == 0.0:
                    continue  # skip no-drug control

                for rep_idx, letter in enumerate(letters):
                    well = f"{letter}{col_num}"
                    bgsub_col = f"{well}_bgsub"
                    if bgsub_col not in df_proc.columns:
                        continue

                    curve = df_proc[bgsub_col].to_numpy().astype(float)

                    # Find first crossing of threshold
                    above = curve >= TTT_THRESHOLD
                    if above.any():
                        idx_first = int(np.argmax(above))
                        if idx_first == 0:
                            ttt_val = float(t_hours[0])
                        else:
                            # Linear interpolation between bounding timepoints
                            t0, t1 = t_hours[idx_first - 1], t_hours[idx_first]
                            y0, y1 = curve[idx_first - 1], curve[idx_first]
                            if y1 != y0:
                                ttt_val = t0 + (TTT_THRESHOLD - y0) / (y1 - y0) * (t1 - t0)
                            else:
                                ttt_val = float(t0)
                        is_censored = False
                    else:
                        ttt_val = t_max
                        is_censored = True

                    ttt_rows.append({
                        "drug": drug_abbrev,
                        "variant": variant_name,
                        "concentration": concentration,
                        "replicate": rep_idx,
                        "well": well,
                        "ttt_hours": ttt_val,
                        "is_censored": is_censored,
                    })

ttt_df = pl.DataFrame(ttt_rows)

# Summary: per variant×drug, median TTT across intermediate concentrations
# Skip lowest 1 and highest 2 concs (where censoring dominates or growth is trivial)
ttt_summary = []
for (drug, variant), grp in ttt_df.group_by(["drug", "variant"]):
    concs_sorted = sorted(grp["concentration"].unique().to_list())
    # Intermediate = skip first and last 2
    if len(concs_sorted) > 3:
        intermediate = concs_sorted[1:-2]
    else:
        intermediate = concs_sorted

    sub = grp.filter(pl.col("concentration").is_in(intermediate))
    censored_rate = sub["is_censored"].sum() / len(sub) if len(sub) > 0 else 1.0

    ttt_summary.append({
        "drug": drug,
        "variant": variant,
        "ttt_median": float(sub["ttt_hours"].median()) if len(sub) > 0 else None,
        "ttt_mean": float(sub["ttt_hours"].mean()) if len(sub) > 0 else None,
        "n_wells": len(sub),
        "censored_rate": float(censored_rate),
    })

ttt_metric_df = pl.DataFrame(ttt_summary)
print(f"TTT per-replicate: {ttt_df.shape}")
print(f"TTT summary: {ttt_metric_df.shape}")
print(f"\nCensoring rates at intermediate concentrations:")
print(ttt_metric_df.sort("drug", "variant").select("drug", "variant", "ttt_median", "censored_rate"))


In [ ]:
# ── Cell B: IC50 at Multiple Timepoints ──
# 4-parameter Hill sigmoid: f(x) = bottom + (top-bottom) / (1 + (x/IC50)^hill)
# Fit OD-vs-concentration at fixed time snapshots (6, 12, 18, 24h).
# Pick best timepoint per drug = lowest mean CV across variants.

from scipy.optimize import curve_fit

def hill_func(x, bottom, top, ic50, hill):
    """4-parameter Hill sigmoid."""
    return bottom + (top - bottom) / (1.0 + (x / ic50) ** hill)

TARGET_HOURS = [6, 12, 18, 24]

ic50_rows = []

for drug_abbrev, experiment in experiments.items():
    concs, _, _ = utl.form_concentrations(experiment)
    n_plates = len(experiment["plates"])

    for plate_idx in range(n_plates):
        plate_variants = experiment["plates"][plate_idx]
        variant_rows_list = experiment["variants"]

        df_raw = utl.read_plate(experiment, plate_idx)
        df_proc = utl.subtract_bg_and_integrate(experiment, df_raw, drop_first=3)
        t_hours = df_proc["t_hours"].to_numpy().astype(float)

        for slot_idx, variant_name in enumerate(plate_variants):
            letters = variant_rows_list[slot_idx]

            for target_h in TARGET_HOURS:
                # Find nearest timepoint
                t_idx = int(np.argmin(np.abs(t_hours - target_h)))
                actual_h = float(t_hours[t_idx])

                for rep_idx, letter in enumerate(letters):
                    # No-drug control OD at this timepoint
                    ctrl_well = f"{letter}11"  # column 11 = no drug
                    ctrl_col = f"{ctrl_well}_bgsub"
                    if ctrl_col not in df_proc.columns:
                        continue
                    ctrl_od = float(df_proc[ctrl_col][t_idx])
                    if ctrl_od <= 0.01:
                        continue  # no growth yet at this timepoint

                    # Collect OD at each drug concentration
                    od_vals = []
                    conc_vals = []
                    for col_idx in range(10):  # columns 1-10 (drug only)
                        col_num = col_idx + 1
                        concentration = concs[col_idx]
                        if concentration <= 0:
                            continue
                        well = f"{letter}{col_num}"
                        bgsub_col = f"{well}_bgsub"
                        if bgsub_col not in df_proc.columns:
                            continue
                        od = float(df_proc[bgsub_col][t_idx])
                        od_vals.append(od / ctrl_od)  # normalize by control
                        conc_vals.append(concentration)

                    if len(conc_vals) < 4:
                        continue

                    conc_arr = np.array(conc_vals)
                    od_arr = np.clip(np.array(od_vals), 0, 2.0)

                    # Fit Hill curve
                    min_conc = conc_arr.min()
                    max_conc = conc_arr.max()
                    try:
                        popt, _ = curve_fit(
                            hill_func, conc_arr, od_arr,
                            p0=[0.0, 1.0, np.sqrt(min_conc * max_conc), 1.0],
                            bounds=(
                                [0.0, 0.5, min_conc / 10, 0.1],
                                [0.5, 1.5, max_conc * 10, 10.0],
                            ),
                            maxfev=5000,
                        )
                        ic50_val = float(popt[2])
                    except (RuntimeError, ValueError):
                        ic50_val = np.nan

                    ic50_rows.append({
                        "drug": drug_abbrev,
                        "variant": variant_name,
                        "target_hour": target_h,
                        "actual_hour": actual_h,
                        "replicate": rep_idx,
                        "well_letter": letter,
                        "ic50_ugml": ic50_val,
                    })

ic50_df = pl.DataFrame(ic50_rows)

# Pick best timepoint per drug: lowest mean CV across variants
ic50_best = {}
print("IC50 by timepoint:")
for drug in ["AZT", "AMP"]:
    print(f"\n  {drug}:")
    best_cv = np.inf
    best_h = None
    for th in TARGET_HOURS:
        sub = ic50_df.filter(
            (pl.col("drug") == drug) & (pl.col("target_hour") == th) & pl.col("ic50_ugml").is_not_nan()
        )
        if len(sub) == 0:
            print(f"    {th}h: no valid fits")
            continue
        # CV per variant
        var_stats = sub.group_by("variant").agg(
            pl.col("ic50_ugml").mean().alias("mean"),
            pl.col("ic50_ugml").std().alias("std"),
        ).with_columns(
            (pl.col("std") / pl.col("mean")).alias("cv")
        )
        mean_cv = var_stats["cv"].drop_nulls().mean()
        n_valid = sub["variant"].n_unique()
        print(f"    {th}h: {n_valid} variants, mean CV={mean_cv:.3f}" if mean_cv else f"    {th}h: {n_valid} variants, CV=N/A")
        if mean_cv is not None and mean_cv < best_cv:
            best_cv = mean_cv
            best_h = th
    ic50_best[drug] = best_h
    print(f"  → Best timepoint: {best_h}h (CV={best_cv:.3f})")

# Build summary at best timepoint
ic50_metric_records = []
for drug in ["AZT", "AMP"]:
    bh = ic50_best[drug]
    if bh is None:
        continue
    sub = ic50_df.filter(
        (pl.col("drug") == drug) & (pl.col("target_hour") == bh) & pl.col("ic50_ugml").is_not_nan()
    )
    for (variant,), vgrp in sub.group_by("variant"):
        vals = vgrp["ic50_ugml"].to_numpy()
        ic50_metric_records.append({
            "drug": drug,
            "variant": variant,
            "ic50_mean": float(np.nanmean(vals)),
            "ic50_std": float(np.nanstd(vals)),
            "ic50_n": int(np.sum(~np.isnan(vals))),
            "best_timepoint_h": bh,
        })

ic50_metric_df = pl.DataFrame(ic50_metric_records)
print(f"\nIC50 summary ({len(ic50_metric_df)} variant×drug pairs):")
print(ic50_metric_df.sort("drug", "variant"))


In [ ]:
# ── Cross-reference MIC with epistasis fitness ──

# Map drug abbreviations to epistasis dataset drug names
DRUG_MAP = {"AZT": "AZT", "AMP": "AMP"}

# Max concentrations in epistasis dataset
MAX_CONC = {"AZT": 324.0, "AMP": 781.0}

xref_records = []

for row in mic_df.iter_rows(named=True):
    drug = row["drug"]
    variant = row["variant"]
    genotype = GENO_LOOKUP.get(variant)

    if genotype is None:
        # DD has no genotype_13
        xref_records.append({
            **row,
            "genotype_13": None,
            "n_mutations": len(variants_meta[variant]["mutations"]),
            "fitness_baseline": None,
            "fitness_max_conc": None,
            "resistance_index": None,
        })
        continue

    # Pull fitness at all concentrations
    fit_rows = epi_df.filter(
        (pl.col("Genotype") == genotype) & (pl.col("Drug") == DRUG_MAP[drug])
    ).sort("Concentration")

    fitness_baseline = fit_rows.filter(pl.col("Concentration") == 0.0)["Fitness"].item()
    fitness_max = fit_rows.filter(pl.col("Concentration") == MAX_CONC[drug])["Fitness"].item()
    resistance_index = fitness_max / fitness_baseline if fitness_baseline != 0 else None

    xref_records.append({
        **row,
        "genotype_13": genotype,
        "n_mutations": len(variants_meta[variant]["mutations"]),
        "fitness_baseline": float(fitness_baseline),
        "fitness_max_conc": float(fitness_max),
        "resistance_index": float(resistance_index) if resistance_index is not None else None,
    })

xref_df = pl.DataFrame(xref_records)

# Add label columns
xref_df = xref_df.with_columns(
    pl.col("variant").map_elements(variant_label, return_dtype=pl.Utf8).alias("label")
)
mic_df = mic_df.with_columns(
    pl.col("variant").map_elements(variant_label, return_dtype=pl.Utf8).alias("label")
)

print("Cross-reference table:")
print(xref_df.sort("drug", "variant"))

In [ ]:
# ── Cell C: PCA from Sequencing Read Counts ──
# SVD on log₂-fold-change matrix: 12 genotypes × (conc × timepoint) features.
# Provides a latent phenotype axis (PC1) that may correlate with resistance better
# than any single concentration.

EPISTASIS_ROOT = ROOT.parent / "Gaszek_Yildiz_Meng_2025"

# ── Load metadata → sample mapping ──
meta_df = pl.read_csv(EPISTASIS_ROOT / "data" / "raw" / "metadata.csv")

# Concentration column has mixed types ("3,100" as string) — clean it
meta_df = meta_df.with_columns(
    pl.col("Concentration").cast(pl.Utf8).str.replace_all(",", "").cast(pl.Float64).alias("Concentration_f")
)

# Filter: AZT/AMP only, skip BLT1, skip t=0 (_to_), skip 3_1mg (irregular)
meta_df = meta_df.filter(
    pl.col("Drug").is_in(["Aztreonam", "Ampicillin"])
    & (pl.col("Timepoint") != "0h")
    & ~pl.col("Sample Name").str.contains("3_1mg")
)
# Map drug names
meta_df = meta_df.with_columns(
    pl.when(pl.col("Drug") == "Aztreonam").then(pl.lit("AZT"))
    .when(pl.col("Drug") == "Ampicillin").then(pl.lit("AMP"))
    .alias("drug_abbrev"),
    # Parse timepoint to numeric hours
    pl.col("Timepoint").str.replace("h", "").cast(pl.Float64).alias("timepoint_h"),
)
print(f"Metadata entries (AZT+AMP, non-zero time): {len(meta_df)}")
print(f"Unique drug×conc×time: {meta_df.select('drug_abbrev','Concentration_f','timepoint_h').unique().shape[0]}")

# ── Build genotype mapping: variant name → dot-notation in read counts ──
geno_to_dot = {}
for name, geno in GENO_LOOKUP.items():
    dots = "".join("." if g == w else g for g, w in zip(geno, WT_GENOTYPE))
    geno_to_dot[name] = dots

dot_to_name = {v: k for k, v in geno_to_dot.items()}
target_dots = list(geno_to_dot.values())
print(f"\nTarget genotype dot-strings ({len(target_dots)}):")
for name, dot in geno_to_dot.items():
    print(f"  {name:6s} → {dot}")

# ── Load read count CSVs ──
drug_file_map = {
    "AZT": "Aztreonam_read_counts_per_genotype.csv",
    "AMP": "Ampicillin_read_counts_per_genotype.csv",
}

pca_records = {}  # {drug: DataFrame with genotypes × features}

for drug_abbrev, fname in drug_file_map.items():
    rc_df = pl.read_csv(EPISTASIS_ROOT / "data" / "raw" / fname)

    # Filter to our 12 genotypes
    rc_sub = rc_df.filter(pl.col("mut_profile_masked").is_in(target_dots))
    print(f"\n{drug_abbrev}: {rc_sub.shape[0]} genotypes matched out of {rc_df.shape[0]} total")

    # Get UT (untreated) columns for this drug
    drug_meta = meta_df.filter(pl.col("drug_abbrev") == drug_abbrev)

    # Separate UT and drug-treated samples
    ut_meta = drug_meta.filter(pl.col("Concentration_f") == 0.0)
    treated_meta = drug_meta.filter(pl.col("Concentration_f") > 0.0)

    # Average UT counts per timepoint across replicates
    ut_by_tp = {}
    for tp in ut_meta["timepoint_h"].unique().to_list():
        tp_samples = ut_meta.filter(pl.col("timepoint_h") == tp)["Sample Name"].to_list()
        valid = [s for s in tp_samples if s in rc_sub.columns]
        if valid:
            ut_by_tp[tp] = rc_sub.select("mut_profile_masked", *valid).with_columns(
                pl.mean_horizontal(*valid).alias(f"ut_{tp}h")
            ).select("mut_profile_masked", f"ut_{tp}h")

    # Build feature matrix: log2 fold-change vs UT at matched timepoint
    feature_cols = []
    feature_names = []

    for row in treated_meta.sort("Concentration_f", "timepoint_h", "Replicate").iter_rows(named=True):
        sample = row["Sample Name"]
        tp = row["timepoint_h"]
        conc = row["Concentration_f"]
        rep = row["Replicate"]

        if sample not in rc_sub.columns:
            continue
        if tp not in ut_by_tp:
            continue

        feature_name = f"C{conc}_{tp}h_R{rep}"
        feature_names.append(feature_name)
        feature_cols.append(sample)

    if not feature_cols:
        print(f"  WARNING: no valid feature columns for {drug_abbrev}")
        continue

    # Extract matrix: genotypes × features (raw counts)
    genotypes = rc_sub["mut_profile_masked"].to_list()
    raw_matrix = rc_sub.select(feature_cols).to_numpy().astype(float)

    # Build UT reference matrix (same timepoint for each feature)
    ut_matrix = np.zeros_like(raw_matrix)
    treated_sorted_rows = list(treated_meta.sort("Concentration_f", "timepoint_h", "Replicate").iter_rows(named=True))
    j = 0
    for row in treated_sorted_rows:
        sample = row["Sample Name"]
        tp = row["timepoint_h"]
        if sample not in rc_sub.columns or tp not in ut_by_tp:
            continue
        ut_col = ut_by_tp[tp].select(f"ut_{tp}h").to_numpy().flatten().astype(float)
        ut_matrix[:, j] = ut_col
        j += 1

    # Log2 fold-change with pseudocount
    pseudocount = 1.0
    lfc = np.log2((raw_matrix + pseudocount) / (ut_matrix + pseudocount))

    # Average across replicates: group by (conc, timepoint)
    ct_keys = []
    for row in treated_sorted_rows:
        if row["Sample Name"] in rc_sub.columns and row["timepoint_h"] in ut_by_tp:
            ct_keys.append((row["Concentration_f"], row["timepoint_h"]))

    unique_ct = sorted(set(ct_keys))
    avg_lfc = np.zeros((len(genotypes), len(unique_ct)))
    avg_names = []
    for k, (c, t) in enumerate(unique_ct):
        indices = [i for i, ct in enumerate(ct_keys) if ct == (c, t)]
        avg_lfc[:, k] = lfc[:, indices].mean(axis=1)
        avg_names.append(f"C{c}_{t}h")

    print(f"  Feature matrix (after rep averaging): {avg_lfc.shape}")

    # Z-score standardize columns
    col_mean = avg_lfc.mean(axis=0)
    col_std = avg_lfc.std(axis=0)
    col_std[col_std == 0] = 1.0
    Z = (avg_lfc - col_mean) / col_std

    # SVD
    U, S, Vt = np.linalg.svd(Z, full_matrices=False)
    variance_explained = (S ** 2) / (S ** 2).sum()

    print(f"  Variance explained: PC1={variance_explained[0]:.1%}, PC2={variance_explained[1]:.1%}, PC3={variance_explained[2]:.1%}")

    # Store PC scores (U * S for proper scaling)
    scores = U * S
    pca_records[drug_abbrev] = pl.DataFrame({
        "variant": [dot_to_name.get(g, g) for g in genotypes],
        "drug": [drug_abbrev] * len(genotypes),
        "PC1": scores[:, 0].tolist(),
        "PC2": scores[:, 1].tolist(),
        "PC3": scores[:, 2].tolist(),
        "var_PC1": [float(variance_explained[0])] * len(genotypes),
        "var_PC2": [float(variance_explained[1])] * len(genotypes),
        "var_PC3": [float(variance_explained[2])] * len(genotypes),
    })

# Combine
pca_df = pl.concat(list(pca_records.values())) if pca_records else pl.DataFrame()
print(f"\nPCA scores: {pca_df.shape}")
if len(pca_df) > 0:
    print(pca_df.sort("drug", "variant"))


In [ ]:
# ── Cell D: Expanded Cross-Reference Table ──
# One row per (variant, drug) with ALL metrics:
#   MIC, TTT, IC50, PC1-PC3, fitness at every epistasis concentration

DRUG_MAP = {"AZT": "AZT", "AMP": "AMP"}
EPI_CONCS = {
    "AZT": epi_df.filter(pl.col("Drug") == "AZT")["Concentration"].unique().sort().to_list(),
    "AMP": epi_df.filter(pl.col("Drug") == "AMP")["Concentration"].unique().sort().to_list(),
}

xref_expanded_records = []

for row in mic_df.iter_rows(named=True):
    drug = row["drug"]
    variant = row["variant"]
    genotype = GENO_LOOKUP.get(variant)

    rec = {
        "drug": drug,
        "variant": variant,
        "label": variant_label(variant),
        "genotype_13": genotype,
        "n_mutations": len(variants_meta[variant]["mutations"]),
        "mic_ugml": row["mic_ugml"],
        "log10_mic": float(np.log10(max(row["mic_ugml"], 1e-3))),
    }

    # ── TTT ──
    ttt_row = ttt_metric_df.filter(
        (pl.col("drug") == drug) & (pl.col("variant") == variant)
    )
    rec["ttt_median"] = float(ttt_row["ttt_median"].item()) if len(ttt_row) > 0 and ttt_row["ttt_median"][0] is not None else None

    # ── IC50 ──
    ic50_row = ic50_metric_df.filter(
        (pl.col("drug") == drug) & (pl.col("variant") == variant)
    )
    if len(ic50_row) > 0 and ic50_row["ic50_mean"][0] is not None:
        ic50_val = float(ic50_row["ic50_mean"].item())
        rec["ic50_mean"] = ic50_val
        rec["log10_ic50"] = float(np.log10(max(ic50_val, 1e-3)))
    else:
        rec["ic50_mean"] = None
        rec["log10_ic50"] = None

    # ── PCA ──
    if len(pca_df) > 0:
        pca_row = pca_df.filter(
            (pl.col("drug") == drug) & (pl.col("variant") == variant)
        )
        for pc in ["PC1", "PC2", "PC3"]:
            rec[pc] = float(pca_row[pc].item()) if len(pca_row) > 0 else None
    else:
        for pc in ["PC1", "PC2", "PC3"]:
            rec[pc] = None

    # ── Fitness at all concentrations ──
    if genotype is not None:
        fit_rows = epi_df.filter(
            (pl.col("Genotype") == genotype) & (pl.col("Drug") == DRUG_MAP[drug])
        ).sort("Concentration")

        fitness_nonzero = []
        for conc in EPI_CONCS[drug]:
            fit_val = fit_rows.filter(pl.col("Concentration") == conc)
            if len(fit_val) > 0:
                f = float(fit_val["Fitness"].item())
                rec[f"fitness_{conc}"] = f
                if conc > 0:
                    fitness_nonzero.append(f)
            else:
                rec[f"fitness_{conc}"] = None

        rec["mean_fitness"] = float(np.mean(fitness_nonzero)) if fitness_nonzero else None
        # resistance_index = fitness at max conc / fitness at 0
        f0 = rec.get("fitness_0.0") or rec.get("fitness_0")
        f_max = rec.get(f"fitness_{EPI_CONCS[drug][-1]}")
        rec["resistance_index"] = float(f_max / f0) if f0 and f_max and f0 != 0 else None
    else:
        for conc in EPI_CONCS[drug]:
            rec[f"fitness_{conc}"] = None
        rec["mean_fitness"] = None
        rec["resistance_index"] = None

    xref_expanded_records.append(rec)

xref_expanded_df = pl.DataFrame(xref_expanded_records)
print(f"Expanded cross-reference: {xref_expanded_df.shape}")
print(f"Columns: {xref_expanded_df.columns}")
print()

# Show non-null counts per column
for col in xref_expanded_df.columns:
    n = xref_expanded_df[col].drop_nulls().len()
    print(f"  {col:25s}: {n}/{len(xref_expanded_df)} non-null")

print("\n" + "="*80)
# Show table for variants with genotypes (exclude DD)
display_cols = ["drug", "variant", "log10_mic", "ttt_median", "log10_ic50", "PC1", "mean_fitness"]
display_cols = [c for c in display_cols if c in xref_expanded_df.columns]
print(xref_expanded_df.filter(pl.col("genotype_13").is_not_null()).select(display_cols).sort("drug", "variant"))


In [ ]:
# ── Altair theme ──

# NOTE: IBM Plex Sans/Mono not installed on this system → vl-convert renders empty text
# Using Avenir Next (Frutiger design) + Menlo (high-quality mono) instead
FONT_BODY = "Avenir Next"
FONT_MONO = "Menlo"

# Variant palette: warm for designed, cool for colonies, black WT, gray DD
# Re-keyed to genotype_13 strings (DD stays "DD")
_RAW_COLORS = {
    "WT":   "#1A1A1A",
    "DD":   "#9E9E9E",
    "v1.1": "#D84315",
    "v1.2": "#E65100",
    "v1.3": "#F57C00",
    "v2.1": "#C62828",
    "v2.2": "#AD1457",
    "v2.3": "#BF360C",
    "v3.1": "#FF8F00",
    "Col1": "#1565C0",
    "Col2": "#0277BD",
    "Col3": "#00838F",
    "Col4": "#00695C",
}
VARIANT_COLORS = {variant_label(k): v for k, v in _RAW_COLORS.items()}

VARIANT_ORDER = [variant_label(v) for v in _NAME_ORDER]

DRUG_COLORS = {"AZT": "#E65100", "AMP": "#1A237E"}

@alt.theme.register("tem1", enable=True)
def tem1_theme():
    return alt.theme.ThemeConfig({
        "config": {
            "font": FONT_BODY,
            "background": "#FAFAFA",
            "view": {"stroke": None},
            "autosize": {"type": "pad"},
            "axis": {
                "labelFont": FONT_BODY,
                "titleFont": FONT_BODY,
                "labelFontSize": 11,
                "titleFontSize": 13,
                "gridColor": "#E8E8E8",
                "domainColor": "#424242",
                "tickColor": "#424242",
            },
            "header": {
                "labelFont": FONT_BODY,
                "titleFont": FONT_BODY,
                "labelFontSize": 12,
                "titleFontSize": 14,
            },
            "legend": {
                "labelFont": FONT_MONO,
                "titleFont": FONT_BODY,
                "labelFontSize": 11,
                "titleFontSize": 12,
            },
            "title": {
                "font": FONT_BODY,
                "fontSize": 15,
                "anchor": "start",
            },
        }
    })

print("Theme registered")
print(f"VARIANT_ORDER: {VARIANT_ORDER}")

In [ ]:
# ── Figure A: Dose-Response Curves ──

# Filter out DD and zero-concentration for log scale
dr_plot = dr_df.filter(
    (pl.col("variant") != "DD") & (pl.col("concentration") > 0)
).to_pandas()

color_domain = [v for v in VARIANT_ORDER if v != "DD"]
color_range = [VARIANT_COLORS[v] for v in color_domain]

base = alt.Chart(dr_plot).encode(
    x=alt.X("concentration:Q", scale=alt.Scale(type="log"), title="Concentration (µg/mL)"),
    color=alt.Color(
        "label:N",
        scale=alt.Scale(domain=color_domain, range=color_range),
        legend=alt.Legend(title="Genotype", columns=4, orient="bottom",
                         labelFont=FONT_MONO, labelFontSize=10),
    ),
)

band = base.mark_area(opacity=0.15).encode(
    y=alt.Y("norm_auc_lo:Q", title="Normalized AUC"),
    y2="norm_auc_hi:Q",
)

line = base.mark_line(strokeWidth=1.8).encode(
    y=alt.Y("norm_auc_mean:Q", title="Normalized AUC"),
)

threshold = alt.Chart(dr_plot).mark_rule(
    strokeDash=[6, 4], strokeWidth=1, color="#616161"
).encode(
    y=alt.datum(MIC_THRESHOLD),
)

fig_a = (
    (band + line + threshold)
    .properties(width=300, height=250)
    .facet(column=alt.Column("drug:N", title=None, header=alt.Header(labelFontSize=14)))
    .resolve_scale(x="independent")
    .properties(title="Dose-Response Curves", padding={"bottom": 10, "right": 10})
)

fig_a.save(str(FIGURES_DIR / "mic_dose_response.png"), scale_factor=2)
fig_a.save(str(FIGURES_DIR / "mic_dose_response.html"))
fig_a

In [ ]:
# ── Figure B: MIC Dot Plot with Error Bars ──
# Log-scale MIC: geometric mean ± 95% CI computed in log10 space
# Layers: whiskers (mark_rule) + replicate dots (small, translucent) + mean dots (large)

from scipy.stats import t as t_dist

# Summary stats in log10 space
mic_summary = (
    mic_rep_df
    .group_by("drug", "label")
    .agg(
        pl.col("log10_mic").mean().alias("mean_log10"),
        pl.col("log10_mic").std().alias("std_log10"),
        pl.col("log10_mic").count().alias("n"),
    )
    .with_columns(
        (pl.col("std_log10") / pl.col("n").sqrt()).alias("sem_log10"),
    )
)

# 95% CI via t-distribution
summary_pd = mic_summary.to_pandas()
summary_pd["ci95"] = summary_pd.apply(
    lambda r: t_dist.ppf(0.975, r["n"] - 1) * r["sem_log10"] if r["n"] > 1 and r["sem_log10"] > 0 else 0, axis=1
)
summary_pd["ci_lo"] = summary_pd["mean_log10"] - summary_pd["ci95"]
summary_pd["ci_hi"] = summary_pd["mean_log10"] + summary_pd["ci95"]

# Sort labels by overall mean MIC descending
label_sort = (
    summary_pd
    .groupby("label")["mean_log10"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)

# Replicate dots
rep_pd = mic_rep_df.to_pandas()

# Shared encodings
x_enc = alt.X("label:N", sort=label_sort, title=None,
              axis=alt.Axis(labelFont=FONT_MONO, labelAngle=-45, labelFontSize=9,
                            labelLimit=200))

y_axis_cfg = alt.Axis(
    labelExpr="pow(10, datum.value) >= 1000 ? format(pow(10, datum.value) / 1000, '.1f') + 'k' : format(pow(10, datum.value), '.1f')",
)

drug_color = alt.Color(
    "drug:N",
    scale=alt.Scale(domain=["AZT", "AMP"], range=[DRUG_COLORS["AZT"], DRUG_COLORS["AMP"]]),
    legend=alt.Legend(title="Drug", orient="top-right"),
)

# Whiskers (95% CI)
whiskers = (
    alt.Chart(summary_pd)
    .mark_rule(strokeWidth=1.5)
    .encode(
        x=x_enc,
        y=alt.Y("ci_lo:Q", title="MIC (µg/mL)", axis=y_axis_cfg),
        y2="ci_hi:Q",
        color=drug_color,
        xOffset="drug:N",
    )
)

# Mean dots (large)
means = (
    alt.Chart(summary_pd)
    .mark_point(size=80, filled=True)
    .encode(
        x=x_enc,
        y=alt.Y("mean_log10:Q", title="MIC (µg/mL)", axis=y_axis_cfg),
        color=drug_color,
        xOffset="drug:N",
    )
)

# Individual replicate dots (small, translucent)
reps = (
    alt.Chart(rep_pd)
    .mark_point(size=20, filled=True, opacity=0.35)
    .encode(
        x=alt.X("label:N", sort=label_sort, title=None,
                axis=alt.Axis(labelFont=FONT_MONO, labelAngle=-45, labelFontSize=9,
                              labelLimit=200)),
        y=alt.Y("log10_mic:Q", title="MIC (µg/mL)", axis=y_axis_cfg),
        color=drug_color,
        xOffset="drug:N",
    )
)

fig_b = (
    (whiskers + reps + means)
    .properties(
        width=550, height=300,
        title="MIC by Variant (geometric mean ± 95% CI)",
        padding={"bottom": 80, "right": 10},
    )
)

fig_b.save(str(FIGURES_DIR / "mic_dotplot.png"), scale_factor=2)
fig_b.save(str(FIGURES_DIR / "mic_dotplot.html"))
fig_b

In [ ]:
# ── Figure C: MIC vs Epistasis Fitness (2-panel: AZT | AMP) ──
# Simple scatter: fitness at max drug concentration vs log10(MIC)
# Regression line computed manually (vl-convert transform_regression unreliable in headless)

import pandas as pd

scatter_df = xref_df.filter(pl.col("genotype_13").is_not_null()).with_columns(
    pl.col("mic_ugml").clip(lower_bound=1e-3).log(base=10).alias("log10_mic"),
)
scatter_pd = scatter_df.to_pandas()

color_domain = [v for v in VARIANT_ORDER if v != "DD"]
color_range = [VARIANT_COLORS[v] for v in color_domain]

color_enc = alt.Color(
    "label:N",
    scale=alt.Scale(domain=color_domain, range=color_range),
    legend=alt.Legend(title="Genotype", columns=4, orient="bottom",
                     labelFont=FONT_MONO, labelFontSize=10),
)

print("Pearson r (fitness @ max conc vs log10 MIC):")
panels = []
for drug in ["AMP", "AZT"]:
    sub_pd = scatter_pd[scatter_pd["drug"] == drug].copy()
    x = sub_pd["fitness_max_conc"].to_numpy()
    y = sub_pd["log10_mic"].to_numpy()
    r, p = stats.pearsonr(x, y)
    print(f"  {drug}: r={r:.3f}, p={p:.4f}, n={len(x)}")

    # Regression line as DataFrame
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.array([x.min(), x.max()])
    y_line = slope * x_line + intercept
    reg_df = pd.DataFrame({"fitness_max_conc": x_line, "log10_mic": y_line})

    base = alt.Chart(sub_pd).encode(
        x=alt.X("fitness_max_conc:Q", title="Fitness (max concentration)"),
        y=alt.Y("log10_mic:Q", title="log₁₀(MIC µg/mL)"),
        color=color_enc,
    )

    points = base.mark_point(size=70, filled=True, opacity=0.85).encode(
        tooltip=[
            alt.Tooltip("label:N", title="Genotype"),
            alt.Tooltip("fitness_max_conc:Q", title="Fitness", format=".3f"),
            alt.Tooltip("mic_ugml:Q", title="MIC (µg/mL)", format=".1f"),
        ],
    )

    reg_line = (
        alt.Chart(reg_df)
        .mark_line(strokeDash=[6, 3], strokeWidth=1.5, opacity=0.5, color="#424242")
        .encode(x="fitness_max_conc:Q", y="log10_mic:Q")
    )

    ann_df = sub_pd.iloc[:1].copy()
    ann_df["annotation"] = f"r = {r:.2f}, p = {p:.3f}"
    ann_df["text_x"] = float(x.min() + (x.max() - x.min()) * 0.02)
    ann_df["text_y"] = float(y.max())

    annotation = (
        alt.Chart(ann_df)
        .mark_text(align="left", baseline="top", fontSize=12,
                   font=FONT_BODY, fontWeight="bold")
        .encode(x=alt.X("text_x:Q"), y=alt.Y("text_y:Q"), text="annotation:N")
    )

    panel = (points + reg_line + annotation).properties(width=300, height=280, title=drug)
    panels.append(panel)

fig_c = (
    alt.hconcat(*panels)
    .resolve_scale(color="shared")
    .properties(
        title="MIC vs Epistasis Fitness (max concentration)",
        padding={"bottom": 10, "right": 10},
    )
)

fig_c.save(str(FIGURES_DIR / "mic_vs_fitness.png"), scale_factor=2)
fig_c.save(str(FIGURES_DIR / "mic_vs_fitness.html"))
fig_c


In [ ]:
# ── Cell E: Correlation Heatmap ──
# Y-axis (experimental metrics): log₁₀(MIC), Time-to-threshold, log₁₀(IC50)
# X-axis (predictors): Mean Fitness, PC1, PC2, F@conc₁, F@conc₂, ...
# Pearson r + significance stars. Two sub-heatmaps: AZT on top, AMP below.

import warnings

# Only use variants with genotypes (exclude DD)
corr_source = xref_expanded_df.filter(pl.col("genotype_13").is_not_null())

Y_METRICS = {
    "log₁₀(MIC)": "log10_mic",
    "TTT (h)": "ttt_median",
    "log₁₀(IC50)": "log10_ic50",
}

def sig_stars(p):
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return ""

corr_records = []

for drug in ["AZT", "AMP"]:
    drug_df = corr_source.filter(pl.col("drug") == drug)
    n = len(drug_df)

    # Build X-metric columns
    x_metrics = {}
    x_metrics["Mean Fitness"] = "mean_fitness"
    if "PC1" in drug_df.columns:
        x_metrics["PC1"] = "PC1"
        x_metrics["PC2"] = "PC2"

    # Fitness at each concentration
    for col in drug_df.columns:
        if col.startswith("fitness_") and col != "fitness_0.0" and col != "fitness_0":
            conc_str = col.replace("fitness_", "")
            x_metrics[f"F@{conc_str}"] = col

    for y_label, y_col in Y_METRICS.items():
        if y_col not in drug_df.columns:
            continue
        y_vals = drug_df[y_col].to_numpy().astype(float)
        y_valid = ~np.isnan(y_vals) & (y_vals is not None)

        for x_label, x_col in x_metrics.items():
            if x_col not in drug_df.columns:
                continue
            x_vals = drug_df[x_col].to_numpy().astype(float)
            x_valid = ~np.isnan(x_vals)

            mask = y_valid & x_valid
            if mask.sum() < 4:
                continue

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                r, p = stats.pearsonr(x_vals[mask], y_vals[mask])

            corr_records.append({
                "drug": drug,
                "y_metric": y_label,
                "x_metric": x_label,
                "r": float(r),
                "p": float(p),
                "n": int(mask.sum()),
                "stars": sig_stars(p),
                "r_text": f"{r:.2f}{sig_stars(p)}",
            })

corr_results_df = pl.DataFrame(corr_records)
print(f"Correlation results: {corr_results_df.shape}")

# ── Plot heatmap ──
corr_pd = corr_results_df.to_pandas()

# Order X-axis: Mean Fitness, PC1, PC2, then fitness concs in order
x_order_base = ["Mean Fitness", "PC1", "PC2"]
x_fitness_cols = sorted(
    [x for x in corr_pd["x_metric"].unique() if x.startswith("F@")],
    key=lambda s: float(s.replace("F@", ""))
)
x_order = [x for x in x_order_base if x in corr_pd["x_metric"].values] + x_fitness_cols
y_order = ["log₁₀(MIC)", "TTT (h)", "log₁₀(IC50)"]

heatmap_panels = []
for drug in ["AZT", "AMP"]:
    sub = corr_pd[corr_pd["drug"] == drug]

    base = alt.Chart(sub).encode(
        x=alt.X("x_metric:N", sort=x_order, title=None,
                axis=alt.Axis(labelAngle=-45, labelFontSize=10)),
        y=alt.Y("y_metric:N", sort=y_order, title=None,
                axis=alt.Axis(labelFontSize=11)),
    )

    rect = base.mark_rect(stroke="#FAFAFA", strokeWidth=1).encode(
        color=alt.Color("r:Q",
            scale=alt.Scale(domain=[-1, 1], scheme="redblue"),
            legend=alt.Legend(title="Pearson r", orient="right"),
        ),
    )

    text = base.mark_text(fontSize=10, font=FONT_MONO).encode(
        text="r_text:N",
        color=alt.condition(
            "abs(datum.r) > 0.65",
            alt.value("white"),
            alt.value("#1A1A1A"),
        ),
    )

    panel = (rect + text).properties(width=400, height=120, title=drug)
    heatmap_panels.append(panel)

fig_e = (
    alt.vconcat(*heatmap_panels)
    .properties(
        title="Correlation: Experimental Metrics vs Epistasis Predictors",
        padding={"bottom": 10, "right": 10},
    )
)

fig_e.save(str(FIGURES_DIR / "mic_correlation_heatmap.png"), scale_factor=2)
fig_e.save(str(FIGURES_DIR / "mic_correlation_heatmap.html"))

# Print best correlations
print("\nTop correlations per drug × Y-metric:")
for drug in ["AZT", "AMP"]:
    for ym in y_order:
        sub = corr_results_df.filter(
            (pl.col("drug") == drug) & (pl.col("y_metric") == ym)
        ).sort(pl.col("r").abs(), descending=True)
        if len(sub) > 0:
            best = sub.row(0, named=True)
            print(f"  {drug} {ym:15s} ← {best['x_metric']:15s}  r={best['r']:.3f}  p={best['p']:.4f}")

fig_e


In [ ]:
# ── Cell F: Best-Pair Scatter Plots ──
# For each drug × Y-metric, find the X-metric with highest |r|.
# 3 rows (Y-metrics) × 2 cols (drugs) scatter grid with regression lines.

import pandas as pd

color_domain = [v for v in VARIANT_ORDER if v != "DD"]
color_range = [VARIANT_COLORS[v] for v in color_domain]

color_enc = alt.Color(
    "label:N",
    scale=alt.Scale(domain=color_domain, range=color_range),
    legend=alt.Legend(title="Genotype", columns=4, orient="bottom",
                     labelFont=FONT_MONO, labelFontSize=10),
)

y_col_map = {"log₁₀(MIC)": "log10_mic", "TTT (h)": "ttt_median", "log₁₀(IC50)": "log10_ic50"}
x_col_map = {"Mean Fitness": "mean_fitness", "PC1": "PC1", "PC2": "PC2"}
for col in xref_expanded_df.columns:
    if col.startswith("fitness_") and col not in ("fitness_0.0", "fitness_0"):
        conc_str = col.replace("fitness_", "")
        x_col_map[f"F@{conc_str}"] = col

scatter_panels = []

for y_label in ["log₁₀(MIC)", "TTT (h)", "log₁₀(IC50)"]:
    y_col = y_col_map[y_label]
    row_panels = []

    for drug in ["AZT", "AMP"]:
        sub_corr = corr_results_df.filter(
            (pl.col("drug") == drug) & (pl.col("y_metric") == y_label)
        ).sort(pl.col("r").abs(), descending=True)

        if len(sub_corr) == 0:
            continue

        best = sub_corr.row(0, named=True)
        x_label = best["x_metric"]
        x_col = x_col_map.get(x_label)
        r_val = best["r"]
        p_val = best["p"]

        if x_col is None:
            continue

        drug_data = xref_expanded_df.filter(
            (pl.col("drug") == drug) & pl.col("genotype_13").is_not_null()
        )

        plot_cols = ["label", x_col, y_col]
        drug_pd = drug_data.select([c for c in plot_cols if c in drug_data.columns]).drop_nulls().to_pandas()

        if len(drug_pd) < 3:
            continue

        # Regression line as pd.DataFrame
        x_arr = drug_pd[x_col].to_numpy().astype(float)
        y_arr = drug_pd[y_col].to_numpy().astype(float)
        slope, intercept = np.polyfit(x_arr, y_arr, 1)
        x_line = np.array([x_arr.min(), x_arr.max()])
        y_line = slope * x_line + intercept
        reg_df = pd.DataFrame({x_col: x_line, y_col: y_line})

        base = alt.Chart(drug_pd).encode(
            x=alt.X(f"{x_col}:Q", title=x_label),
            y=alt.Y(f"{y_col}:Q", title=y_label),
            color=color_enc,
        )

        points = base.mark_point(size=70, filled=True, opacity=0.85)

        reg = (
            alt.Chart(reg_df)
            .mark_line(strokeDash=[6, 3], strokeWidth=1.5, opacity=0.5, color="#424242")
            .encode(x=f"{x_col}:Q", y=f"{y_col}:Q")
        )

        ann_text = f"r={r_val:.2f}, p={p_val:.3f}"
        ann_data = drug_pd.iloc[:1].copy()
        ann_data["ann"] = ann_text
        ann_data["ax"] = float(x_arr.min() + (x_arr.max() - x_arr.min()) * 0.02)
        ann_data["ay"] = float(y_arr.max())

        ann = (
            alt.Chart(ann_data)
            .mark_text(align="left", baseline="top", fontSize=11,
                       font=FONT_BODY, fontWeight="bold")
            .encode(x="ax:Q", y="ay:Q", text="ann:N")
        )

        panel = (points + reg + ann).properties(
            width=250, height=220,
            title=f"{drug}: {x_label}"
        )
        row_panels.append(panel)

    if row_panels:
        scatter_panels.append(alt.hconcat(*row_panels))

if scatter_panels:
    fig_f = (
        alt.vconcat(*scatter_panels)
        .resolve_scale(color="shared")
        .properties(
            title="Best Correlating Metric Pairs",
            padding={"bottom": 10, "right": 10},
        )
    )

    fig_f.save(str(FIGURES_DIR / "mic_best_correlations.png"), scale_factor=2)
    fig_f.save(str(FIGURES_DIR / "mic_best_correlations.html"))
    fig_f
else:
    print("WARNING: No scatter panels to display (insufficient data)")


In [ ]:
# ── Figure D: Growth Curves Small Multiples ──
# Fix: v1.1 had 2 curves/conc (dual slots on plate 7) → collect per-replicate, then aggregate
# Fix: DD excluded (dead mutant, flat noise)

SMOOTH_WINDOW = 6

# High-contrast categorical palette (warm/cool alternation, visible on #FAFAFA)
CONC_PALETTE = ["#E63946", "#457B9D", "#2A9D8F", "#E9C46A", "#264653", "#F4A261"]
CONC_DASHES = [[1, 0], [8, 4], [4, 4], [8, 2, 2, 2], [2, 2], [12, 4]]

# Variant order excluding DD
variant_order_no_dd = [v for v in VARIANT_ORDER if v != "DD"]

for drug_abbrev in ["AZT", "AMP"]:
    experiment = experiments[drug_abbrev]
    concs, _, _ = utl.form_concentrations(experiment)
    gc_rows = []

    for plate_idx in range(len(experiment["plates"])):
        plate_variants = experiment["plates"][plate_idx]
        variant_rows_list = experiment["variants"]

        df_raw = utl.read_plate(experiment, plate_idx)
        df_proc = utl.subtract_bg_and_integrate(experiment, df_raw, drop_first=3)
        t_hours = df_proc["t_hours"].to_numpy().astype(float)

        for slot_idx, variant_name in enumerate(plate_variants):
            if variant_name == "DD":
                continue  # exclude dead mutant

            letters = variant_rows_list[slot_idx]

            for col_idx in range(11):  # columns 1-11 only
                col_num = col_idx + 1
                concentration = concs[col_idx]
                if concentration == 0.0:
                    continue  # skip no-drug control

                # Collect individual replicate curves (not averaged)
                for letter in letters:
                    well = f"{letter}{col_num}"
                    bgsub_col = f"{well}_bgsub"
                    if bgsub_col not in df_proc.columns:
                        continue

                    curve = df_proc[bgsub_col].to_numpy().astype(float)

                    # Smooth per-replicate
                    if len(curve) >= SMOOTH_WINDOW:
                        kernel = np.ones(SMOOTH_WINDOW) / SMOOTH_WINDOW
                        smoothed = np.convolve(curve, kernel, mode="valid")
                        t_smooth = t_hours[SMOOTH_WINDOW - 1:]
                    else:
                        smoothed = curve
                        t_smooth = t_hours

                    for i in range(len(smoothed)):
                        gc_rows.append({
                            "variant": variant_name,
                            "label": variant_label(variant_name),
                            "concentration": concentration,
                            "time_h": t_smooth[i],
                            "od_bgsub": smoothed[i],
                            "rep_id": f"{plate_idx}_{letter}",
                        })

    # Aggregate: mean across replicates per (variant, label, concentration, time_h)
    # This merges v1.1's 6 replicates (3 per slot on plate 7) into one clean curve
    gc_df = (
        pl.DataFrame(gc_rows)
        .group_by(["variant", "label", "concentration", "time_h"])
        .agg(pl.col("od_bgsub").mean().alias("od_bgsub"))
        .sort("variant", "concentration", "time_h")
    ).to_pandas()

    # Pick ~6 representative concentrations (log-spaced)
    all_concs = sorted([c for c in gc_df["concentration"].unique() if c > 0])
    if len(all_concs) > 6:
        indices = np.linspace(0, len(all_concs) - 1, 6, dtype=int)
        selected_concs = [all_concs[i] for i in indices]
    else:
        selected_concs = all_concs
    gc_plot = gc_df[gc_df["concentration"].isin(selected_concs)].copy()

    # Categorical concentration string for nominal encoding
    gc_plot["conc_str"] = gc_plot["concentration"].apply(
        lambda c: f"{c:.0f}" if c >= 1 else f"{c:.2f}"
    )
    conc_str_order = [f"{c:.0f}" if c >= 1 else f"{c:.2f}" for c in selected_concs]

    fig_d = (
        alt.Chart(gc_plot)
        .mark_line(strokeWidth=1.4)
        .encode(
            x=alt.X("time_h:Q", title="Time (h)"),
            y=alt.Y("od_bgsub:Q", title="OD₆₀₀ (bg-subtracted)"),
            color=alt.Color(
                "conc_str:N",
                scale=alt.Scale(domain=conc_str_order, range=CONC_PALETTE[:len(conc_str_order)]),
                legend=alt.Legend(title="Conc (µg/mL)", orient="bottom", columns=6),
            ),
            strokeDash=alt.StrokeDash(
                "conc_str:N",
                scale=alt.Scale(domain=conc_str_order, range=CONC_DASHES[:len(conc_str_order)]),
                legend=None,
            ),
        )
        .properties(width=200, height=150)
        .facet(
            facet=alt.Facet("label:N", title=None, sort=variant_order_no_dd,
                           header=alt.Header(labelFont=FONT_MONO, labelFontSize=10)),
            columns=4,
        )
        .resolve_scale(y="independent")
        .properties(
            title=f"Growth Curves — {drug_abbrev}",
            padding={"bottom": 10, "right": 10},
        )
    )

    fname = f"growth_curves_{drug_abbrev.lower()}"
    fig_d.save(str(FIGURES_DIR / f"{fname}.png"), scale_factor=2)
    fig_d.display()

print("Growth curve figures saved")

In [ ]:
# ── Summary table + CSV export ──

print("="*80)
print("FULL CROSS-REFERENCE TABLE")
print("="*80)
summary = xref_df.select(
    "variant", "drug", "mic_ugml", "n_mutations",
    "fitness_baseline", "fitness_max_conc", "resistance_index",
).sort("drug", "mic_ugml", descending=[False, True])
print(summary)

print("\n" + "="*80)
print("DD NEGATIVE CONTROL")
print("="*80)
dd = mic_df.filter(pl.col("variant") == "DD")
print(dd)

print("\n" + "="*80)
print("WT MIC")
print("="*80)
wt = mic_df.filter(pl.col("variant") == "WT")
print(wt)

# Export
out_path = ROOT / "src" / "mic_summary.csv"
xref_df.write_csv(out_path)
print(f"\nSaved: {out_path}")

# Expanded cross-reference (all metrics)
expanded_path = ROOT / "src" / "mic_expanded_summary.csv"
xref_expanded_df.write_csv(expanded_path)
print(f"Saved: {expanded_path}")

# Correlation results
corr_path = ROOT / "src" / "metric_correlations.csv"
corr_results_df.write_csv(corr_path)
print(f"Saved: {corr_path}")
